# Dataset Splitting

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

DATA_FILE = "ml_dataset.csv"

def prepare():

    df = pd.read_csv(DATA_FILE)

    # -------- FEATURE SELECTION --------
    features = [
        "mean_snr",
        "sat_count",
        "mean_elevation",
        "mean_azimuth",
        "roll",
        "pitch",
        "yaw",
        "inclination_angle",
        "position_jump"
    ]

    # Targets
    target_x = "error_x"
    target_y = "error_y"

    # Combine features and targets for NaN handling
    all_relevant_columns = features + [target_x, target_y]

    # Drop rows with any NaN values in the relevant columns
    df.dropna(subset=all_relevant_columns, inplace=True)

    X = df[features]
    y_x = df[target_x]
    y_y = df[target_y]

    # -------- SPLIT --------
    X_train, X_test, yx_train, yx_test, yy_train, yy_test = train_test_split(
        X, y_x, y_y, test_size=0.3, random_state=42
    )

    # -------- SCALING (for DL & XGB stability) --------
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Save everything
    joblib.dump((X_train_scaled, X_test_scaled, yx_train, yx_test, yy_train, yy_test), "data.pkl")
    joblib.dump(scaler, "scaler.pkl")

    print("✅ Data prepared and saved")

if __name__ == "__main__":
    prepare()

✅ Data prepared and saved


# Random Forest Algorithm Training

In [2]:
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

X_train, X_test, yx_train, yx_test, yy_train, yy_test = joblib.load("data.pkl")

model_x = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model_y = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)

model_x.fit(X_train, yx_train)
model_y.fit(X_train, yy_train)

pred_x = model_x.predict(X_test)
pred_y = model_y.predict(X_test)

rmse_x = np.sqrt(mean_squared_error(yx_test, pred_x))
rmse_y = np.sqrt(mean_squared_error(yy_test, pred_y))

print("🌳 Random Forest Results")
print(f"RMSE X: {rmse_x:.2f}")
print(f"RMSE Y: {rmse_y:.2f}")

joblib.dump(model_x, "rf_x.pkl")
joblib.dump(model_y, "rf_y.pkl")

🌳 Random Forest Results
RMSE X: 1.24
RMSE Y: 1.79


['rf_y.pkl']

# XGBoost Algorithm Training

In [ ]:
import joblib
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error

# -------- LOAD DATA --------
X_train, X_test, yx_train, yx_test, yy_train, yy_test = joblib.load("data.pkl")

# -------- PARAM GRID --------
param_dist = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 5, 10]
}

# -------- BASE MODEL --------
xgb = XGBRegressor(random_state=42)

# -------- RANDOM SEARCH --------
search_x = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=25,
    scoring="neg_mean_squared_error",
    cv=3,
    verbose=1,
    n_jobs=-1
)

search_y = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=25,
    scoring="neg_mean_squared_error",
    cv=3,
    verbose=1,
    n_jobs=-1
)

print("🔍 Tuning X model...")
search_x.fit(X_train, yx_train)

print("🔍 Tuning Y model...")
search_y.fit(X_train, yy_train)

# -------- BEST MODELS --------
best_x = search_x.best_estimator_
best_y = search_y.best_estimator_

# -------- PREDICTION --------
pred_x = best_x.predict(X_test)
pred_y = best_y.predict(X_test)

rmse_x = np.sqrt(mean_squared_error(yx_test, pred_x))
rmse_y = np.sqrt(mean_squared_error(yy_test, pred_y))

print("\n🚀 TUNED XGBOOST RESULTS")
print(f"RMSE X: {rmse_x:.2f}")
print(f"RMSE Y: {rmse_y:.2f}")

print("\nBest Params (X):", search_x.best_params_)
print("Best Params (Y):", search_y.best_params_)

# -------- SAVE --------
joblib.dump(best_x, "xgb_x.pkl")
joblib.dump(best_y, "xgb_y.pkl")

🔍 Tuning X model...
Fitting 3 folds for each of 25 candidates, totalling 75 fits
🔍 Tuning Y model...
Fitting 3 folds for each of 25 candidates, totalling 75 fits

🚀 TUNED XGBOOST RESULTS
RMSE X: 1.32
RMSE Y: 1.89

Best Params (X): {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
Best Params (Y): {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 100, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bytree': 0.6}


['xgb_tuned_y.pkl']

# SVM Algorithm Training


In [4]:
import joblib
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

# -------- LOAD DATA --------
X_train, X_test, yx_train, yx_test, yy_train, yy_test = joblib.load("data.pkl")

# -------- MODEL --------
# Using RBF kernel (best for non-linear data)

model_x = SVR(kernel='rbf', C=50, gamma='scale', epsilon=0.05)
model_y = SVR(kernel='rbf', C=50, gamma='scale', epsilon=0.05)

print("Training SVM...")

model_x.fit(X_train, yx_train)
model_y.fit(X_train, yy_train)

# -------- PREDICTION --------
pred_x = model_x.predict(X_test)
pred_y = model_y.predict(X_test)

# -------- EVALUATION --------
rmse_x = np.sqrt(mean_squared_error(yx_test, pred_x))
rmse_y = np.sqrt(mean_squared_error(yy_test, pred_y))

print("\n🟣 SVM Results")
print(f"RMSE X: {rmse_x:.2f}")
print(f"RMSE Y: {rmse_y:.2f}")

# -------- SAVE --------
joblib.dump(model_x, "svm_x.pkl")
joblib.dump(model_y, "svm_y.pkl")

Training SVM...

🟣 SVM Results
RMSE X: 2.00
RMSE Y: 2.52


['svm_y.pkl']

# Create Sequence Dataset for LSTM and TCN

In [18]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler

DATA_FILE = "ml_dataset.csv"
SEQ_LEN = 8  # slightly longer window helps

FEATURES = [
    "mean_snr","sat_count","mean_elevation","mean_azimuth",
    "roll","pitch","yaw","inclination_angle","position_jump"
]

def make_sequences(df):
    # ensure strict time order - commented out as 'timestamp' column is not available
    # df = df.sort_values("timestamp").reset_index(drop=True)

    X = df[FEATURES].values
    y = df[["error_x","error_y"]].values

    X_seq, y_seq = [], []
    for i in range(len(df) - SEQ_LEN):
        X_seq.append(X[i:i+SEQ_LEN])
        y_seq.append(y[i+SEQ_LEN])
    return np.array(X_seq), np.array(y_seq)

def main():
    df = pd.read_csv(DATA_FILE)

    # Combine features and targets for NaN handling
    all_relevant_columns = FEATURES + ["error_x", "error_y"]

    # Drop rows with any NaN values in the relevant columns
    df.dropna(subset=all_relevant_columns, inplace=True)

    # IMPORTANT: if you have multiple runs, group by run/file id
    # If not available, this still works but grouping is better.
    # Example (if you had a column 'run_id'):
    # groups = [g for _, g in df.groupby("run_id")]
    groups = [df]

    X_all, y_all = [], []
    for g in groups:
        Xs, ys = make_sequences(g)
        X_all.append(Xs); y_all.append(ys)

    X_all = np.vstack(X_all)
    y_all = np.vstack(y_all)

    # train/test split (time-aware split preferred)
    split = int(0.7 * len(X_all))
    X_train, X_test = X_all[:split], X_all[split:]
    y_train, y_test = y_all[:split], y_all[split:]

    # -------- SCALE FEATURES --------
    scaler_x = StandardScaler()
    # fit on flattened time axis
    X_train_2d = X_train.reshape(-1, X_train.shape[-1])
    scaler_x.fit(X_train_2d)

    def scale_seq(X, scaler):
        X2 = X.reshape(-1, X.shape[-1])
        X2 = scaler.transform(X2)
        return X2.reshape(X.shape)

    X_train = scale_seq(X_train, scaler_x)
    X_test  = scale_seq(X_test,  scaler_x)

    # -------- SCALE TARGETS (helps DL) --------
    scaler_y = StandardScaler()
    scaler_y.fit(y_train)
    y_train_s = scaler_y.transform(y_train)
    y_test_s  = scaler_y.transform(y_test)

    joblib.dump((X_train, X_test, y_train_s, y_test_s, scaler_y), "seq_data_fixed.pkl")
    joblib.dump(scaler_x, "seq_scaler_x.pkl")

    print("✅ Fixed sequence data ready")

if __name__ == "__main__":
    main()

✅ Fixed sequence data ready


# LSTM Training

In [22]:
import joblib
import numpy as np
from tensorflow import keras
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test, scaler_y = joblib.load("seq_data_fixed.pkl")

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1], X_train.shape[2])),

    keras.layers.LSTM(64, return_sequences=True),
    keras.layers.Dropout(0.2),

    keras.layers.LSTM(32),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(2)
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
]

print("Training LSTM (tuned)...")
model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# -------- PREDICT & INVERSE SCALE --------
pred_s = model.predict(X_test)
pred = scaler_y.inverse_transform(pred_s)
y_true = scaler_y.inverse_transform(y_test)

rmse_x = np.sqrt(mean_squared_error(y_true[:,0], pred[:,0]))
rmse_y = np.sqrt(mean_squared_error(y_true[:,1], pred[:,1]))

print("\n🟣 Tuned LSTM Results")
print(f"RMSE X: {rmse_x:.2f}")
print(f"RMSE Y: {rmse_y:.2f}")

model.save("lstm_tuned.h5")

Training LSTM (tuned)...
Epoch 1/15
577/577 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.3046 - val_loss: 0.9964 - learning_rate: 0.0010
Epoch 2/15
577/577 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 0.1754 - val_loss: 1.0939 - learning_rate: 0.0010
Epoch 3/15
577/577 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 0.1593 - val_loss: 0.9834 - learning_rate: 0.0010
Epoch 4/15
577/577 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 0.1479 - val_loss: 1.1182 - learning_rate: 0.0010
Epoch 5/15
577/577 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.1403 - val_loss: 1.0608 - learning_rate: 0.0010
Epoch 6/15
577/577 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 0.1336 - val_loss: 1.0014 - learning_rate: 0.0010
Epoch 7/15
577/577 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 0.1217 - val_loss: 1.0515 - learning_rate: 5.0000e-04
Epoch 8/15
577/577 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - loss: 0.1180 - val_loss: 0.9516 - learning_rate: 5.0000e-04
Epoch 9/15
577/577 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 0.1149 - val_loss: 0.9


🟣 Tuned LSTM Results
RMSE X: 4.06
RMSE Y: 9.10


# TCN Training

In [ ]:
import joblib
import numpy as np
from tcn import TCN
from tensorflow import keras
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = joblib.load("seq_data.pkl")

model = keras.Sequential([
    TCN(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(2)
])

model.compile(optimizer='adam', loss='mse')

print("Training TCN...")
model.fit(X_train, y_train, epochs=20, batch_size=32)

pred = model.predict(X_test)

rmse_x = np.sqrt(mean_squared_error(y_test[:,0], pred[:,0]))
rmse_y = np.sqrt(mean_squared_error(y_test[:,1], pred[:,1]))

print("\n🟣 TCN Results")
print(f"RMSE X: {rmse_x:.2f}")
print(f"RMSE Y: {rmse_y:.2f}")

model.save("tcn_model.h5")

/usr/local/lib/python3.12/dist-packages/tcn/tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


Training TCN...
Epoch 1/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 19s 9ms/step - loss: 455.8654
Epoch 2/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 22.0303
Epoch 3/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 25.4872
Epoch 4/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 17.4631
Epoch 5/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 17.1432
Epoch 6/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 16.3091
Epoch 7/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 15.1620
Epoch 8/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 12.8104
Epoch 9/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 12.7619
Epoch 10/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 10.0422
Epoch 11/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 9.7662 
Epoch 12/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 9.1799
Epoch 13/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 8.5881
Epoch 14/20
721/721 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 7.7833
Epoch 15/20
721/721 ━━━━━━━━


🟣 TCN Results
RMSE X: 3.86
RMSE Y: 9.97


# Model Evaluation

In [ ]:
import joblib
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, confusion_matrix
)

from tensorflow import keras

# -------- LOAD TABULAR DATA --------
X_train, X_test, yx_train, yx_test, yy_train, yy_test = joblib.load("data.pkl")

# -------- LOAD SEQUENCE DATA --------
X_seq_train, X_seq_test, y_seq_train, y_seq_test, scaler_y = joblib.load("seq_data_fixed.pkl")

# -------- RESULT STORAGE --------
results = []

# -------- EVALUATION FUNCTION --------
def evaluate_model(name, pred_x, pred_y, yx_true, yy_true):

    # -------- REGRESSION METRICS --------
    rmse_x = np.sqrt(mean_squared_error(yx_true, pred_x))
    rmse_y = np.sqrt(mean_squared_error(yy_true, pred_y))

    mae_x = mean_absolute_error(yx_true, pred_x)
    mae_y = mean_absolute_error(yy_true, pred_y)

    r2_x = r2_score(yx_true, pred_x)
    r2_y = r2_score(yy_true, pred_y)

    # -------- ERROR MAGNITUDE --------
    true_error = np.sqrt(yx_true**2 + yy_true**2)
    pred_error = np.sqrt(pred_x**2 + pred_y**2)

    # -------- CLASSIFICATION (threshold) --------
    threshold = 2.0

    true_class = (true_error < threshold).astype(int)
    pred_class = (pred_error < threshold).astype(int)

    acc = accuracy_score(true_class, pred_class)
    prec = precision_score(true_class, pred_class, zero_division=0)
    rec = recall_score(true_class, pred_class, zero_division=0)

    cm = confusion_matrix(true_class, pred_class)

    print(f"\n📊 {name} Results")
    print(f"RMSE X: {rmse_x:.2f} | RMSE Y: {rmse_y:.2f}")
    print(f"MAE X: {mae_x:.2f} | MAE Y: {mae_y:.2f}")
    print(f"R² X: {r2_x:.2f} | R² Y: {r2_y:.2f}")
    print(f"Accuracy: {acc:.2f} | Precision: {prec:.2f} | Recall: {rec:.2f}")
    print("Confusion Matrix:\n", cm)

    results.append({
        "Model": name,
        "RMSE_X": rmse_x,
        "RMSE_Y": rmse_y,
        "MAE_X": mae_x,
        "MAE_Y": mae_y,
        "R2_X": r2_x,
        "R2_Y": r2_y,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec
    })


# ================================
# 🔷 RANDOM FOREST
# ================================
rf_x = joblib.load("rf_x.pkl")
rf_y = joblib.load("rf_y.pkl")

pred_x = rf_x.predict(X_test)
pred_y = rf_y.predict(X_test)

evaluate_model("Random Forest", pred_x, pred_y, yx_test, yy_test)


# ================================
# 🔷 XGBOOST (TUNED)
# ================================
xgb_x = joblib.load("xgb_tuned_x.pkl")
xgb_y = joblib.load("xgb_tuned_y.pkl")

pred_x = xgb_x.predict(X_test)
pred_y = xgb_y.predict(X_test)

evaluate_model("Tuned XGBoost", pred_x, pred_y, yx_test, yy_test)


# ================================
# 🔷 SVM
# ================================
svm_x = joblib.load("svm_x.pkl")
svm_y = joblib.load("svm_y.pkl")

pred_x = svm_x.predict(X_test)
pred_y = svm_y.predict(X_test)

evaluate_model("SVM", pred_x, pred_y, yx_test, yy_test)


# ================================
# 🔷 LSTM
# ================================
lstm = keras.models.load_model("lstm_tuned.h5", compile=False)

pred = lstm.predict(X_seq_test)

# inverse scaling
pred = scaler_y.inverse_transform(pred)
y_true = scaler_y.inverse_transform(y_seq_test)

evaluate_model("LSTM", pred[:,0], pred[:,1], y_true[:,0], y_true[:,1])


# ================================
# 🔷 TCN
# ================================
from tcn import TCN

tcn_model = keras.models.load_model("tcn_model.h5", custom_objects={"TCN": TCN}, compile=False)

pred = tcn_model.predict(X_seq_test)

pred = scaler_y.inverse_transform(pred)
y_true = scaler_y.inverse_transform(y_seq_test)

evaluate_model("TCN", pred[:,0], pred[:,1], y_true[:,0], y_true[:,1])


# ================================
# 🔷 SAVE RESULTS
# ================================
df = pd.DataFrame(results)
df.to_csv("model_comparison_results.csv", index=False)

print("\n✅ All model evaluations saved to model_comparison_results.csv")


📊 Random Forest Results
RMSE X: 1.24 | RMSE Y: 1.79
MAE X: 0.52 | MAE Y: 0.73
R² X: 0.90 | R² Y: 0.91
Accuracy: 0.99 | Precision: 0.95 | Recall: 0.67
Confusion Matrix:
 [[9539   12]
 [ 110  226]]

📊 Tuned XGBoost Results
RMSE X: 1.32 | RMSE Y: 1.89
MAE X: 0.52 | MAE Y: 0.76
R² X: 0.89 | R² Y: 0.90
Accuracy: 0.99 | Precision: 0.91 | Recall: 0.88
Confusion Matrix:
 [[9521   30]
 [  42  294]]

📊 SVM Results
RMSE X: 2.00 | RMSE Y: 2.52
MAE X: 0.92 | MAE Y: 1.14
R² X: 0.74 | R² Y: 0.82
Accuracy: 0.99 | Precision: 0.91 | Recall: 0.85
Confusion Matrix:
 [[9524   27]
 [  51  285]]
309/309 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

📊 LSTM Results
RMSE X: 4.06 | RMSE Y: 9.10
MAE X: 2.96 | MAE Y: 8.33
R² X: -1.12 | R² Y: -2.04
Accuracy: 0.71 | Precision: 0.00 | Recall: 0.00
Confusion Matrix:
 [[7034 1799]
 [1051    0]]
309/309 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step

📊 TCN Results
RMSE X: 3.98 | RMSE Y: 8.34
MAE X: 2.87 | MAE Y: 6.89
R² X: -1.04 | R² Y: -1.55
Accuracy: 0.89 | Precision: 0.00 | Recall: 0.00
Conf